In [6]:
import pandas as pd
from datetime import date 
from pathlib import Path
from functools import reduce

DATA_DIR = Path('/workspace/data/')
LAST_EXPORT = date.today().strftime("%d_%m_%Y")
FILE = DATA_DIR / f"{LAST_EXPORT}.xls"

if not FILE.exists():
    FILE = list(DATA_DIR.glob("*.xls"))[-1]


months_pt = {
    'janeiro': 'January',
    'fevereiro': 'February',
    'março': 'March',
    'abril': 'April',
    'maio': 'May',
    'junho': 'June',
    'julho': 'July',
    'agosto': 'August',
    'setembro': 'September',
    'outubro': 'October',
    'novembro': 'November',
    'dezembro': 'December'
}

FILE

PosixPath('/workspace/data/28_10_2025.xls')

In [7]:
df = pd.read_excel(FILE)
df.dropna(axis='index',how='all',inplace=True)
df.rename(columns={'Unnamed: 0':'target'}, inplace=True)
del df['Dispositivo']

dropRow = df[df['target'].str.contains('Uso Total')]
dropCol = df.columns.get_loc('Uso Total')

df_clean = df.loc[ : dropRow.index[0] - 1 ].iloc[:, : dropCol].copy()
# df_clean

df_T = df_clean.set_index('target').T

df_T.index = (
    df_T.index.map(lambda x: x.lower())
    .to_series()
    .replace(months_pt, regex=True)
    .pipe(pd.to_datetime, format='%d de %B de %Y')
)

df_T.fillna('0s', inplace=True)
df_T

target,∞ ENERGY,10.254.211.203,100.80.21.72,192,192.168.1.2,192.168.1.254,1banda.com,3dcuritiba.com.br,3dlab.com.br,3m.com.br,...,Zedge,Zepp,zepp.com,ZFC,Zizians,zobnin.github.io,zorin.com,Zotero,zotero.org,zotfile.com
2025-02-18,0s,0s,0s,0s,0s,0s,0s,0s,0s,0s,...,0s,0s,0s,0s,0s,0s,0s,0s,0s,0s
2025-02-19,0s,0s,0s,0s,0s,0s,0s,0s,0s,0s,...,0s,0s,0s,0s,0s,0s,0s,0s,0s,0s
2025-02-20,0s,0s,0s,0s,0s,0s,0s,0s,0s,0s,...,0s,0s,0s,0s,0s,0s,0s,0s,0s,0s
2025-02-21,0s,0s,0s,0s,0s,0s,0s,0s,0s,0s,...,0s,0s,0s,0s,0s,0s,0s,0s,0s,0s
2025-02-22,0s,0s,0s,0s,0s,0s,0s,0s,0s,0s,...,0s,0s,0s,0s,0s,0s,0s,0s,0s,0s
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-10-24,0s,0s,0s,0s,0s,0s,0s,0s,0s,0s,...,0s,55s,0s,0s,0s,0s,0s,0s,0s,0s
2025-10-25,0s,0s,0s,0s,0s,0s,0s,0s,0s,0s,...,0s,2m 12s,0s,0s,0s,0s,0s,0s,0s,0s
2025-10-26,0s,0s,0s,0s,0s,0s,0s,0s,0s,0s,...,0s,37m 54s,0s,0s,0s,0s,0s,0s,0s,0s
2025-10-27,0s,0s,0s,0s,0s,0s,0s,0s,0s,0s,...,0s,1m 51s,0s,0s,0s,0s,0s,0s,0s,0s


In [8]:
def parser_time(string_split: str) -> int:
    string_split = str(string_split)
    try:
        if 's' in string_split:
            return int(string_split.replace('s',''))
        elif 'm' in string_split:
            return int(string_split.replace('m','')) * 60
        elif 'h' in string_split:
            return int(string_split.replace('h','')) * 60 * 60
        else:
            return int(string_split)
    except:
        raise Exception(f"Invalid split time date {string_split}")

def time2sec(time_in: str) -> int:
    return sum(parser_time(i) for i in time_in.split())

In [9]:
df_secs = df_T.map(time2sec).copy()

df_final = df_secs.T.groupby(df_secs.columns).sum().T
df_final

target,10.254.211.203,100.80.21.72,192,192.168.1.2,192.168.1.254,1banda.com,3dcuritiba.com.br,3dlab.com.br,3m.com.br,3pitech.com,...,ynab.com,youtube.com,ytdlp,z2jh.jupyter.org,zepp.com,zobnin.github.io,zorin.com,zotero.org,zotfile.com,∞ ENERGY
2025-02-18,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2025-02-19,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2025-02-20,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2025-02-21,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2025-02-22,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-10-24,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2025-10-25,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2025-10-26,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2025-10-27,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [10]:
df_final.to_csv(DATA_DIR/"df_final.csv", quoting=1)
df_final.to_pickle(DATA_DIR/"df_final.pkl")